# EZStats - Train Event Spotter on Ball Action Spotting
Run order: A -> B -> C -> D -> E -> F (update classes after D!) -> G -> H -> I

## Cell A - Install + Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import subprocess
subprocess.run(['apt-get', 'install', '-y', '-q', 'p7zip-full'], check=True)
subprocess.run(['pip', 'install', '-q', 'huggingface_hub', 'torch', 'torchvision'], check=True)

from pathlib import Path
SN_BAS_ROOT   = Path('/content/drive/MyDrive/ezstats/sn-bas-2025')
LOCAL_EXTRACT = Path('/content/sn-bas-extracted')
FEAT_DIR      = Path('/content/drive/MyDrive/ezstats/sn-bas-features')
for p in [SN_BAS_ROOT, LOCAL_EXTRACT, FEAT_DIR]:
    p.mkdir(parents=True, exist_ok=True)
print('Ready.')

## Cell B - Download SN-BAS-2025 from HuggingFace
Public dataset, no login needed. Skip if already downloaded.

In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path

SN_BAS_ROOT = Path('/content/drive/MyDrive/ezstats/sn-bas-2025')

if list(SN_BAS_ROOT.glob('*.zip')):
    print('Already downloaded - skipping.')
else:
    try:
        snapshot_download(
            repo_id='SoccerNet/SN-BAS-2025',
            repo_type='dataset',
            revision='main',
            local_dir=str(SN_BAS_ROOT),
        )
        print('Download complete:', SN_BAS_ROOT)
    except Exception as e:
        if '401' in str(e) or '403' in str(e) or 'gated' in str(e).lower():
            from huggingface_hub import login
            login()
            snapshot_download(
                repo_id='SoccerNet/SN-BAS-2025',
                repo_type='dataset',
                revision='main',
                local_dir=str(SN_BAS_ROOT),
            )
            print('Download complete:', SN_BAS_ROOT)
        else:
            raise

## Cell C - Extract zips
Uses 7z (handles AES-256). Extracts train.zip + valid.zip to /content/ for fast I/O.

In [ ]:
    import subprocess
    from pathlib import Path

    SN_BAS_ROOT   = Path('/content/drive/MyDrive/ezstats/sn-bas-2025')
    LOCAL_EXTRACT = Path('/content/sn-bas-extracted')
    LOCAL_EXTRACT.mkdir(exist_ok=True)

    for zip_name in ['train.zip', 'valid.zip']:
        zf = SN_BAS_ROOT / zip_name
        if not zf.exists():
            print(f'SKIP (not on Drive): {zip_name}')
            continue
        existing = list(LOCAL_EXTRACT.rglob('Labels-ball.json'))
        if existing and zip_name == 'train.zip':
            print(f'Already extracted: {zip_name} ({len(existing)} games found)')
            continue
        print(f'Extracting {zip_name} ({zf.stat().st_size/1024/1024/1024:.1f} GB)...')
        result = subprocess.run(
            ['7z', 'x', '-ps0cc3rn3t', f'-o{LOCAL_EXTRACT}', '-y', str(zf)],
            capture_output=True, text=True
        )
        if result.returncode > 1:
            print(f'ERROR (code {result.returncode}):', result.stdout[-400:])
        else:
            print(f'  Done: {zip_name}')

    labels = sorted(LOCAL_EXTRACT.rglob('Labels-ball.json'))
    videos = sorted(LOCAL_EXTRACT.rglob('*224p.mp4')) + sorted(LOCAL_EXTRACT.rglob('*224p.mkv'))
    print(f'Extracted: {len(labels)} label files, {len(videos)} 224p videos')
    for l in labels[:5]:
        print(' ', l.relative_to(LOCAL_EXTRACT))

## Cell D - Find all labels + videos
Searches both local extracted dir and Drive HF download dir.

In [ ]:
from pathlib import Path

SN_BAS_ROOT   = Path('/content/drive/MyDrive/ezstats/sn-bas-2025')
LOCAL_EXTRACT = Path('/content/sn-bas-extracted')

game_dirs = {}
for root in [LOCAL_EXTRACT, SN_BAS_ROOT]:
    for lf in sorted(root.rglob('Labels-ball.json')):
        game_dir  = lf.parent
        game_name = game_dir.name
        if game_name not in game_dirs:
            vids = (
                sorted(game_dir.glob('*224p.mp4')) +
                sorted(game_dir.glob('*224p.mkv')) +
                sorted(game_dir.glob('1_224p*')) +
                sorted(game_dir.glob('2_224p*'))
            )
            game_dirs[game_name] = {'label': lf, 'videos': list(set(vids))}

print(f'Games with labels: {len(game_dirs)}')
games_with_video = {k: v for k, v in game_dirs.items() if v['videos']}
print(f'Games with labels + 224p video: {len(games_with_video)}')
for name, info in list(game_dirs.items())[:8]:
    vnames = [v.name for v in info['videos']]
    print(f'  {name[:50]}')
    print(f'    label : {info["label"].name}')
    print(f'    videos: {vnames}')

if not game_dirs:
    print('No labels found - make sure Cell C completed successfully.')

## Cell E - Inspect label format
Read the output carefully - confirm class names and position unit before Cell F.

In [ ]:
import json
from pathlib import Path

SN_BAS_ROOT   = Path('/content/drive/MyDrive/ezstats/sn-bas-2025')
LOCAL_EXTRACT = Path('/content/sn-bas-extracted')

label_files = sorted(LOCAL_EXTRACT.rglob('Labels-ball.json')) + sorted(SN_BAS_ROOT.rglob('Labels-ball.json'))
if not label_files:
    print('No label files - run Cell C first.')
else:
    data = json.loads(label_files[0].read_text())
    print('Top-level keys:', list(data.keys()))
    anns = data.get('annotations', data.get('events', []))
    print(f'Annotations: {len(anns)}')
    print('First 5:')
    for a in anns[:5]:
        print(' ', a)
    all_labels = set()
    for f in label_files:
        d = json.loads(f.read_text())
        for a in d.get('annotations', d.get('events', [])):
            all_labels.add(a.get('label', a.get('type', '')))
    print('\nAll unique classes:', sorted(all_labels))
    print('\nSample positions:', [a.get('position') for a in anns[:5]])
    print('Sample gameTimes:', [a.get('gameTime') for a in anns[:5]])

## Cell F - Extract ResNet18 features
Saves to Drive so features survive Colab disconnects. Skip-safe.

In [ ]:
import cv2, numpy as np, torch, torch.nn as nn
from pathlib import Path
from torchvision import models, transforms

SN_BAS_ROOT   = Path('/content/drive/MyDrive/ezstats/sn-bas-2025')
LOCAL_EXTRACT = Path('/content/sn-bas-extracted')
FEAT_DIR      = Path('/content/drive/MyDrive/ezstats/sn-bas-features')
FEAT_DIR.mkdir(parents=True, exist_ok=True)
FEAT_FPS = 2.0

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
backbone.fc = nn.Identity()
backbone = backbone.to(device).eval()

preprocess = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def extract_features(video_path):
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    interval = max(1, int(round(fps / FEAT_FPS)))
    feats, idx = [], 0
    with torch.no_grad():
        while True:
            ret, frame = cap.read()
            if not ret: break
            if idx % interval == 0:
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                t   = preprocess(rgb).unsqueeze(0).to(device)
                f   = backbone(t).squeeze(0).cpu().numpy()
                norm = np.linalg.norm(f)
                feats.append(f / norm if norm > 0 else f)
            idx += 1
    cap.release()
    return np.array(feats, dtype=np.float32), fps

video_label_pairs = []
for root in [LOCAL_EXTRACT, SN_BAS_ROOT]:
    for lf in sorted(root.rglob('Labels-ball.json')):
        game_dir = lf.parent
        for vid in sorted(game_dir.glob('*224p*')):
            if vid.suffix in ('.mp4', '.mkv'):
                video_label_pairs.append((vid, lf))

print(f'Found {len(video_label_pairs)} video+label pairs')

for video_path, label_path in video_label_pairs:
    game_name = video_path.parent.name
    safe_name = game_name.replace(' ', '_').replace('/', '_')
    feat_path = FEAT_DIR / f'{safe_name}_{video_path.stem}.npy'
    fps_path  = FEAT_DIR / f'{safe_name}_{video_path.stem}.fps.txt'
    if feat_path.exists():
        print(f'  SKIP: {game_name[:40]} / {video_path.name}')
        continue
    print(f'  {game_name[:40]} / {video_path.name}...', end='', flush=True)
    feats, src_fps = extract_features(video_path)
    np.save(str(feat_path), feats)
    fps_path.write_text(str(src_fps))
    print(f' {len(feats)} frames @ {src_fps:.1f}fps')

print('Done.')

## Cell G - Build dataset
Update BALL_ACTION_CLASSES to match Cell E output before running.

In [ ]:
import json, numpy as np
from pathlib import Path
from collections import Counter

SN_BAS_ROOT   = Path('/content/drive/MyDrive/ezstats/sn-bas-2025')
LOCAL_EXTRACT = Path('/content/sn-bas-extracted')
FEAT_DIR      = Path('/content/drive/MyDrive/ezstats/sn-bas-features')
FEAT_FPS      = 2.0
WINDOW        = 15

# Update after Cell E shows actual class names
BALL_ACTION_CLASSES = [
    'background', 'PASS', 'DRIVE', 'HEADER', 'HIGH PASS',
    'OUT', 'CORNER', 'CROSS', 'THROW IN', 'SHOT',
    'BALL PLAYER BLOCK', 'PLAYER SUCCESSFUL TACKLE', 'FREE KICK', 'GOAL',
]
CLASS_TO_IDX = {c.upper(): i for i, c in enumerate(BALL_ACTION_CLASSES)}
NUM_CLASSES  = len(BALL_ACTION_CLASSES)
print(f'{NUM_CLASSES} classes')

def pos_to_feat_frame(ann, src_fps):
    pos = ann.get('position', 0)
    return int(pos / 1000 * FEAT_FPS) if pos > 100000 else int(pos / src_fps * FEAT_FPS)

def parse_half(gt):
    try: return int(gt.split(' - ')[0].strip())
    except: return -1

def build_samples(feat_path, fps_path, label_path, half, window=15, radius=1, bg_keep=0.04):
    feats   = np.load(str(feat_path))
    src_fps = float(fps_path.read_text().strip()) if fps_path.exists() else 25.0
    T       = len(feats)
    labels  = np.zeros(T, dtype=np.int64)
    data    = json.loads(label_path.read_text())
    for ann in data.get('annotations', data.get('events', [])):
        if parse_half(ann.get('gameTime', '')) != half:
            continue
        cls = CLASS_TO_IDX.get(ann.get('label', ann.get('type', '')).upper(), 0)
        if cls == 0: continue
        ff = pos_to_feat_frame(ann, src_fps)
        for k in range(max(0, ff - radius), min(T, ff + radius + 1)):
            labels[k] = cls
    samples = []
    for i in range(T - window + 1):
        c = labels[i + window // 2]
        if c == 0 and np.random.rand() > bg_keep: continue
        samples.append((feats[i:i+window].copy(), int(c)))
    return samples

np.random.seed(42)

label_by_game = {}
for root in [LOCAL_EXTRACT, SN_BAS_ROOT]:
    for lf in sorted(root.rglob('Labels-ball.json')):
        safe = lf.parent.name.replace(' ', '_').replace('/', '_')
        if safe not in label_by_game:
            label_by_game[safe] = lf

pairs = []
for feat_path in sorted(FEAT_DIR.glob('*.npy')):
    fps_path = feat_path.with_suffix('.fps.txt')
    stem = feat_path.stem
    half = 2 if '_2_224p' in stem or stem.endswith('_2') else 1
    label_path = None
    for safe, lf in label_by_game.items():
        if stem.startswith(safe):
            label_path = lf
            break
    if label_path:
        pairs.append((feat_path, fps_path, label_path, half))
    else:
        print(f'  WARNING: no label for {stem[:60]}')

print(f'Matched pairs: {len(pairs)}')
split       = max(1, int(len(pairs) * 0.75))
train_pairs = pairs[:split]
val_pairs   = pairs[split:]

def collect(pair_list):
    samples = []
    for fp, fpsp, lp, h in pair_list:
        s = build_samples(fp, fpsp, lp, h)
        samples.extend(s)
        print(f'  {lp.parent.name[:40]} h{h}: {len(s)} samples')
    return samples

train_samples = collect(train_pairs)
val_samples   = collect(val_pairs)

counts = Counter(s[1] for s in train_samples)
print('\nClass distribution (train):')
for i, name in enumerate(BALL_ACTION_CLASSES):
    if counts[i]: print(f'  {i:2d}  {name:<28} {counts[i]:>6}')
print(f'\nTrain: {len(train_samples)}, Val: {len(val_samples)}')

## Cell H - Train Bi-LSTM

In [ ]:
import torch, torch.nn as nn, numpy as np
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class EventDataset(Dataset):
    def __init__(self, s):
        self.X = torch.tensor(np.array([x[0] for x in s]), dtype=torch.float32)
        self.y = torch.tensor([x[1] for x in s], dtype=torch.long)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

total    = len(train_samples)
weights  = [min(total / (NUM_CLASSES * max(counts[i], 1)), 50.0) for i in range(NUM_CLASSES)]
w_tensor = torch.tensor(weights, dtype=torch.float32).to(device)

train_loader = DataLoader(EventDataset(train_samples), batch_size=256, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(EventDataset(val_samples),   batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

class EventSpotter(nn.Module):
    def __init__(self, n):
        super().__init__()
        self.lstm = nn.LSTM(512, 256, batch_first=True, bidirectional=True, num_layers=2, dropout=0.3)
        self.head = nn.Sequential(nn.Linear(512, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, n))
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, out.shape[1] // 2])

model     = EventSpotter(NUM_CLASSES).to(device)
criterion = nn.CrossEntropyLoss(weight=w_tensor)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5, verbose=True)

best_val_loss, best_state = float('inf'), None

for epoch in range(1, 41):
    model.train()
    tl = tc = tt = 0
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(X)
        loss   = criterion(logits, y)
        loss.backward()
        optimizer.step()
        tl += loss.item()*len(y); tc += (logits.argmax(1)==y).sum().item(); tt += len(y)

    model.eval()
    vl = vc = vt = 0
    with torch.no_grad():
        for X, y in val_loader:
            X, y = X.to(device), y.to(device)
            out = model(X)
            vl += criterion(out, y).item()*len(y); vc += (out.argmax(1)==y).sum().item(); vt += len(y)

    scheduler.step(vl/vt)
    flag = ''
    if vl/vt < best_val_loss:
        best_val_loss = vl/vt
        best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        flag = ' <- best'
    print(f'Ep {epoch:02d} | train {tl/tt:.4f} acc={tc/tt:.3f} | val {vl/vt:.4f} acc={vc/vt:.3f}{flag}')

## Cell I - Per-class accuracy on val

In [ ]:
from collections import defaultdict
model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
model.eval()
pc, pt = defaultdict(int), defaultdict(int)
with torch.no_grad():
    for X, y in val_loader:
        X, y = X.to(device), y.to(device)
        preds = model(X).argmax(1)
        for t, p in zip(y.cpu().numpy(), preds.cpu().numpy()):
            pt[t] += 1
            if t == p: pc[t] += 1
print('Per-class val accuracy:')
for i, name in enumerate(BALL_ACTION_CLASSES):
    if pt[i]:
        print(f'  {name:<28} {pc[i]/pt[i]:.2f}  ({pc[i]}/{pt[i]})')

## Cell J - Save best model to Drive

In [ ]:
import json, torch
from pathlib import Path

save_dir = Path('/content/drive/MyDrive/ezstats/runs/event_spotter_bas2025')
save_dir.mkdir(parents=True, exist_ok=True)
torch.save(best_state, str(save_dir / 'model.pt'))
(save_dir / 'classes.json').write_text(json.dumps(BALL_ACTION_CLASSES, indent=2))
print(f'Saved -> {save_dir}/model.pt')
print('Next steps on laptop:')
print('  1. Download model.pt -> artifacts/training/event_spotter_bas2025/model.pt')
print('  2. Update event_spotter.py: SOCCERNET_CLASSES -> BALL_ACTION_CLASSES, head 18->14')
print('  3. Update run scripts: --event-model-name artifacts/training/event_spotter_bas2025/model.pt')